In [4]:
import numpy as np
import pandas as pd 
import pickle
from pathlib import Path
from astropy.timeseries import LombScargle
from scipy.stats import skew, kurtosis, shapiro
from utils.preprocessing import fourier_features, stetson_K, fourier_fit

# Paths
# Using absolute path to data directory to avoid FileNotFoundError when running from notebooks/
DATA_DIR = Path("/home/admin/main/ucsd-phys-139-final/data")
FEATURES_PATH = DATA_DIR / "features.csv"
LABELED_FEATURES_PATH = DATA_DIR / "labeled_features.csv"
TESS_PKL = DATA_DIR / "time_flux_pdcsap.pkl"

# Cross-match paths
GAIA_MATCH = DATA_DIR / "gaia_crossmatch/gaia_crossmatch.csv"
ASASSN_MATCH = DATA_DIR / "asassn/asassn_crossmatch.csv"

In [5]:
def compute_features(light_curves):

    """
    Computes features for list of light curves.
    Matches the logic in preprocessing.ipynb exactly.
    """
    
    rows = []
    
    print(f"Processing {len(light_curves)} light curves...")
    
    for idx, item in enumerate(light_curves):
        # Support either (t, f) or (star_id, t, f)
        if isinstance(item, (list, tuple)) and len(item) == 3:
            star_id, t, f = item
        else:
            t, f = item
            # If star_id is not provided, we use index. 
            # WARNING: This assumes the pickle order matches the filename order used for cross-matching.
            # Ideally, the pickle should contain (filename, t, f).
            star_id = idx 

        # remove NaNs
        mask = np.isfinite(t) & np.isfinite(f)
        t, f = t[mask], f[mask]

        # --- Period (Lomb–Scargle) ---
        try:
            freq, power = LombScargle(t, f).autopower()
            best_period = 1 / freq[np.argmax(power)]
        except Exception:
            best_period = np.nan

        # --- Flux distribution features ---
        if len(f) > 0:
            Q1 = np.percentile(f, 25)
            Q3 = np.percentile(f, 75)
            Q31 = Q3 - Q1
            Std = np.std(f)
            gamma1 = skew(f)
            gamma2 = kurtosis(f, fisher=True)
            W, _ = shapiro(f)
            K = stetson_K(f)

            # --- Fourier features ---
            R21, R31, phi21, phi31, Amp = fourier_features(best_period, t, f)
        else:
            # Handle empty/bad lightcurves
            Q31 = Std = gamma1 = gamma2 = W = K = R21 = R31 = phi21 = phi31 = Amp = np.nan

        # store (ensure star_id is first key)
        rows.append({
            "star_id": star_id,
            "period": best_period,
            "Q31": Q31,
            "Amp": Amp,
            "W": W,
            "K": K,
            "Std": Std,
            "gamma1": gamma1,
            "gamma2": gamma2,
            "R21": R21,
            "R31": R31,
            "phi21": phi21,
            "phi31": phi31
        })

        if idx % 1000 == 0:
            print(f"Processed {idx}/{len(light_curves)}")

    df = pd.DataFrame(rows)
    # Reorder columns explicitly to keep star_id first
    ordered_cols = [
        "star_id", "period", "Q31", "Amp", "W", "K", "Std",
        "gamma1", "gamma2", "R21", "R31", "phi21", "phi31"
    ]
    return df[ordered_cols]

In [6]:
# 1. Load Light Curves and Compute Features
if not TESS_PKL.exists():
    print(f"Pickle not found: {TESS_PKL}")
    # If pickle doesn't exist, we might need to recreate it from FITS or rely on existing features.csv if it exists
    # For this script, we'll assume we want to regenerate features.csv to be safe or load it if it exists.
    if FEATURES_PATH.exists():
        print(f"Loading existing features from {FEATURES_PATH}")
        df_features = pd.read_csv(FEATURES_PATH)
    else:
        # One final check: maybe data is in ../data relative to notebooks?
        # But we used absolute path above, so if it fails, the file is truly missing.
        raise FileNotFoundError(f"Neither pickle ({TESS_PKL}) nor features.csv ({FEATURES_PATH}) found.")
else:
    print(f"Loading light curves from {TESS_PKL}...")
    with TESS_PKL.open("rb") as f:
        light_curves = pickle.load(f)
    
    # Compute features
    df_features = compute_features(light_curves)
    
    # Save the standard features.csv (without labels) as requested first
    df_features.to_csv(FEATURES_PATH, index=False)
    print(f"Saved basic features to {FEATURES_PATH}")

Loading light curves from /home/admin/main/ucsd-phys-139-final/data/time_flux_pdcsap.pkl...
Processing 2976 light curves...
Processed 0/2976


/home/admin/miniforge3/envs/phys139/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 14746.
  res = hypotest_fun_out(*samples, **kwds)
/home/admin/miniforge3/envs/phys139/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 14776.
  res = hypotest_fun_out(*samples, **kwds)
/home/admin/miniforge3/envs/phys139/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 14853.
  res = hypotest_fun_out(*samples, **kwds)
/home/admin/miniforge3/envs/phys139/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 14545.
  res = hypotest_fun_out(*samples, **kw

KeyboardInterrupt: 

In [ ]:
# 2. Load Cross-Match Results

# Note: The key challenge here is linking the 'star_id' from the pickle/features to the cross-match results.
# The cross-match results (ASASSN/Gaia) use TESS filenames or RA/Dec.
# The current preprocessing.py assigns 'star_id' as the integer index of the list.
# Unless the pickle stores (filename, t, f), we can't reliably join on ID.
# Assuming the pickle was generated in the same order as the file list, or contains identifiers.
# Let's inspect the first element of light_curves if we loaded it.

has_filenames = False
if 'light_curves' in locals() and len(light_curves) > 0:
    item = light_curves[0]
    if len(item) == 3:
        print("Pickle contains (id, t, f). Using ID for matching.")
        has_filenames = True
    else:
        print("Pickle contains only (t, f). WARNING: Cannot reliably match to external catalogs without IDs.")
        print("Creating dummy labels for demonstration logic.")
        
# For now, we will proceed. If we have filenames in star_id, we can match.
# If star_id is just an index (0, 1, 2...), we can't match to ASASSN/Gaia filenames unless 
# we know the mapping.

# Check if cross-match files exist
gaia_df = pd.read_csv(GAIA_MATCH) if GAIA_MATCH.exists() else pd.DataFrame()
asassn_df = pd.read_csv(ASASSN_MATCH) if ASASSN_MATCH.exists() else pd.DataFrame()

print(f"Loaded {len(gaia_df)} Gaia matches")
print(f"Loaded {len(asassn_df)} ASASSN matches")

In [ ]:
# 3. Add Label Column
# We want to add a label column. 
# Logic: If a star is in ASASSN or Gaia (with a variability flag/class), we label it.
# Or perhaps we just want to merge the external class info.

# Create a 'label' or 'class' column. Default to 'Unknown' or 0.
df_features['label'] = 'Unknown'

if has_filenames:
    # Assuming star_id in df_features is the filename or matches TESS_Filename in cross-match
    # We need to ensure string format matches (e.g. trimming extensions)
    
    # Merge ASASSN Class
    if not asassn_df.empty:
        # Map TESS filename to ASASSN Class
        # ASASSN crossmatch has 'TESS file name' and 'ASASSN Class'
        asassn_map = dict(zip(asassn_df['TESS file name'], asassn_df['ASASSN Class']))
        
        # Update labels where we have a match
        df_features['label'] = df_features['star_id'].map(asassn_map).fillna(df_features['label'])
        
    # Merge Gaia Class (if available and needed)
    if not gaia_df.empty and 'Gaia_Class' in gaia_df.columns:
        # Map TESS Filename to Gaia Class. 
        # Priority: ASASSN > Gaia? Or just overwrite? 
        # Let's keep existing label if found (ASASSN priority), otherwise use Gaia.
        gaia_map = dict(zip(gaia_df['TESS_Filename'], gaia_df['Gaia_Class']))
        
        # Only update if still Unknown
        mask_unknown = df_features['label'] == 'Unknown'
        new_labels = df_features.loc[mask_unknown, 'star_id'].map(gaia_map)
        df_features.loc[mask_unknown, 'label'] = new_labels.fillna('Unknown')

# Save the labeled dataset
# Same columns as features.csv + 'label'
df_features.to_csv(LABELED_FEATURES_PATH, index=False)
print(f"Saved labeled features to {LABELED_FEATURES_PATH}")
print("Sample of labeled data:")
print(df_features[df_features['label'] != 'Unknown'].head())